In [1]:
!pip install -q mlflow dagshub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 110.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 123.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.3/273.3 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 105.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [2]:
import os
import re
import json
import random
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.callbacks import (EarlyStopping, ModelCheckpoint,ReduceLROnPlateau)
import mlflow
import dagshub

In [3]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

os.environ["PYTHONHASHSEED"] = str(SEED)

In [4]:
mlflow.set_tracking_uri("https://dagshub.com/Ahmad450-gcu/Next-Word-Predictor.mlflow")

In [5]:
import dagshub
dagshub.init(repo_owner='Ahmad450-gcu', repo_name='Next-Word-Predictor', mlflow=True)
mlflow.set_experiment("WikiText2_Baseline_Preprocessing")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=b7b64653-018e-4ee1-9e18-17c95e89d7ad&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=e4673cb40c75f738de385c1ee8e364a938ab7a56ada1b38c6a1fa4947c482f00




Accessing as Ahmad450-gcu

Initialized MLflow to track repo "Ahmad450-gcu/Next-Word-Predictor"

Repository Ahmad450-gcu/Next-Word-Predictor initialized!

<Experiment: artifact_location='mlflow-artifacts:/bd5eb7154e254e0983676b72654e1157', creation_time=1785056889924, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1785056889924, lifecycle_stage='active', name='WikiText2_Baseline_Preprocessing', tags={}, trace_location=None, workspace='default'>

In [6]:
mlflow.tensorflow.autolog(log_models=True, checkpoint=False)

In [7]:
config = {

    # Preprocessing
    "remove_whitespace": True,
    "remove_headings": False,
    "lowercase": False,
    "normalize_wikitext": False,
    "min_frequency": 5,

    # Sequence Generation
    "sequence_length": 20,

    # Model
    "embedding_dim": 128,
    "lstm_units": 256,

    # Training
    "batch_size": 64,
    "epochs": 10,
    "learning_rate": 0.001,

    # Random Seed
    "seed": SEED
}

In [8]:
mlflow.start_run(run_name="Experiment_01_Baseline")
mlflow.log_params(config)

In [9]:
# Loading dataset
with open("wiki.train.raw", "r", encoding="utf-8") as f:
    train_data = f.readlines()
with open("wiki.valid.raw", "r", encoding="utf-8") as f:
    valid_data = f.readlines()
with open("wiki.test.raw", "r", encoding="utf-8") as f:
    test_data = f.readlines()
print(f"Train : {len(train_data):,} lines")
print(f"Valid : {len(valid_data):,} lines")
print(f"Test  : {len(test_data):,} lines")

Train : 36,718 lines
Valid : 3,760 lines
Test  : 4,358 lines


In [10]:
# split corpus into individual articles with whitespaces as seperators
def split_articles(lines):
    articles = []
    current_article = []
    for line in lines:
        line = line.strip()
        if line == "":
            if current_article:
                articles.append(current_article)
                current_article = []
        else:
            current_article.append(line)
    if current_article:
        articles.append(current_article)
    return articles

In [11]:
train_articles = split_articles(train_data)
valid_articles = split_articles(valid_data)
test_articles = split_articles(test_data)

In [12]:
print(f"Train Articles : {len(train_articles):,}")
print(f"Valid Articles : {len(valid_articles):,}")
print(f"Test Articles  : {len(test_articles):,}")

Train Articles : 11,669
Valid Articles : 1,160
Test Articles  : 1,320


In [13]:
mlflow.log_metric("train_articles", len(train_articles))
mlflow.log_metric("valid_articles", len(valid_articles))
mlflow.log_metric("test_articles", len(test_articles))

In [14]:
def articles_to_text(articles):
    article_texts = []
    for article in articles:
        text = " ".join(article)
        article_texts.append(text)
    return article_texts

In [15]:
train_texts = articles_to_text(train_articles)
valid_texts = articles_to_text(valid_articles)
test_texts = articles_to_text(test_articles)

In [16]:
print("Number of training articles:", len(train_texts))
print("\nFirst training article:\n")
print(train_texts[:10][:100])   # Print first 1000 characters

Number of training articles: 11669

First training article:

['= Valkyria Chronicles III =', 'Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustm

In [17]:
train_characters = sum(len(text) for text in train_texts)
valid_characters = sum(len(text) for text in valid_texts)
test_characters = sum(len(text) for text in test_texts)
print(f"Train Characters : {train_characters:,}")
print(f"Valid Characters : {valid_characters:,}")
print(f"Test Characters  : {test_characters:,}")

Train Characters : 10,833,784
Valid Characters : 1,136,065
Test Characters  : 1,278,520


In [18]:
mlflow.log_metrics({
    "train_articles": len(train_texts),
    "valid_articles": len(valid_texts),
    "test_articles": len(test_texts),
    "train_characters": train_characters,
    "valid_characters": valid_characters,
    "test_characters": test_characters
})

In [19]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(lower=False, filters="", oov_token="<UNK>")
tokenizer.fit_on_texts(train_texts)

In [68]:
vocab_size = len(tokenizer.word_index) + 1
print(f"Vocabulary Size: {vocab_size:,}")
print("\nFirst 20 Vocabulary Entries:\n")
for word, idx in list(tokenizer.word_index.items())[:20]:
    print(f"{idx:>5} : {word}")

Vocabulary Size: 76,618

First 20 Vocabulary Entries:

    1 : <UNK>
    2 : the
    3 : ,
    4 : .
    5 : of
    6 : and
    7 : in
    8 : to
    9 : a
   10 : =
   11 : "
   12 : was
   13 : The
   14 : @-@
   15 : that
   16 : as
   17 : 's
   18 : on
   19 : for
   20 : with


In [71]:
mlflow.log_param("lowercase", False)
mlflow.log_param("filters", "")
mlflow.log_param("oov_token", "<UNK>")
mlflow.log_metric("vocabulary_size", vocab_size)

In [72]:
print("=" * 23)
print("Vocabulary Statistics")
print("=" * 23)
print(f"Vocabulary Size : {vocab_size}")

Vocabulary Statistics
Vocabulary Size : 76618


In [73]:
word_counts = tokenizer.word_counts
top_20 = sorted(word_counts.items(), key=lambda x: x[1],reverse=True)[:20]

In [74]:
top20_df = (pd.DataFrame(top_20, columns=["Word", "Frequency"]))
top20_df

,Word,Frequency
0,the,113161
1,",",99913
2,.,73388
3,of,56889
4,and,50603
5,in,39453
6,to,39190
7,a,34237
8,=,29570
9,"""",28309


In [75]:
freq_df = pd.DataFrame(
    word_counts.items(),
    columns=["Word", "Frequency"]
)

freq_df["Frequency"].describe()

,Frequency
count,76616.000000
mean,26.781743
std,756.143991
min,1.000000
25%,1.000000
50%,2.000000
75%,6.000000
max,113161.000000


In [76]:
singleton_words = sum(1 for count in word_counts.values() if count == 1)
rare_words = sum(1 for count in word_counts.values() if count <= 5)

print(f"Singleton Words : {singleton_words:,}")
print(f"Words ≤5 Frequency : {rare_words:,}")

Singleton Words : 32,055
Words ≤5 Frequency : 55,990


In [77]:
print("OOV Token:")
print(tokenizer.oov_token)
print("\nOOV Index:")
print(tokenizer.word_index[tokenizer.oov_token])

OOV Token:
<UNK>

OOV Index:
1


In [78]:
mlflow.log_metrics({
    "vocabulary_size": vocab_size,
    "singleton_words": singleton_words,
    "rare_words_leq5": rare_words
})

In [79]:
with open("tokenizer_word_index.json", "w") as f:
    json.dump(tokenizer.word_index,f,indent=4)
mlflow.log_artifact("tokenizer_word_index.json")

In [80]:
train_sequences = tokenizer.texts_to_sequences(train_texts)
valid_sequences = tokenizer.texts_to_sequences(valid_texts)
test_sequences = tokenizer.texts_to_sequences(test_texts)

In [81]:
print("Encoded Sequence:")
print(train_sequences[10][:100])

Encoded Sequence:
[10, 10, 10, 4205, 10, 10, 10]


In [82]:
train_total_tokens = sum(len(seq) for seq in train_sequences)
valid_total_tokens = sum(len(seq) for seq in valid_sequences)
test_total_tokens = sum(len(seq) for seq in test_sequences)

print(f"Train Tokens : {train_total_tokens:,}")
print(f"Valid Tokens : {valid_total_tokens:,}")
print(f"Test Tokens  : {test_total_tokens:,}")

Train Tokens : 2,051,910
Valid Tokens : 213,886
Test Tokens  : 241,211


In [83]:
oov_index = tokenizer.word_index[tokenizer.oov_token]

train_oov = sum(token == oov_index for seq in train_sequences for token in seq)
valid_oov = sum(token == oov_index for seq in valid_sequences for token in seq)
test_oov = sum(token == oov_index for seq in test_sequences for token in seq)

print(f"Train OOV Tokens : {train_oov:,}")
print(f"Valid OOV Tokens : {valid_oov:,}")
print(f"Test OOV Tokens  : {test_oov:,}")

Train OOV Tokens : 0
Valid OOV Tokens : 7,183
Test OOV Tokens  : 9,864


In [84]:
train_oov_pct = (train_oov / train_total_tokens) * 100
valid_oov_pct = (valid_oov / valid_total_tokens) * 100
test_oov_pct = (test_oov / test_total_tokens) * 100

print(f"Train OOV % : {train_oov_pct:.4f}%")
print(f"Valid OOV % : {valid_oov_pct:.4f}%")
print(f"Test OOV %  : {test_oov_pct:.4f}%")

Train OOV % : 0.0000%
Valid OOV % : 3.3583%
Test OOV %  : 4.0894%


In [85]:
mlflow.log_metrics({
    "train_total_tokens": train_total_tokens,
    "valid_total_tokens": valid_total_tokens,
    "test_total_tokens": test_total_tokens,
    "train_oov_percent": train_oov_pct,
    "valid_oov_percent": valid_oov_pct,
    "test_oov_percent": test_oov_pct
})

In [86]:
def generate_sequences(sequences, sequence_length):
    X = []
    y = []
    for article in sequences:
        if len(article) <= sequence_length:
            continue
        for i in range(sequence_length, len(article)):
            X.append(article[i-sequence_length:i])
            y.append(article[i])
    return X, y

In [87]:
SEQUENCE_LENGTH = config["sequence_length"]
X_train, y_train = generate_sequences(train_sequences, SEQUENCE_LENGTH)
X_valid, y_valid = generate_sequences(valid_sequences, SEQUENCE_LENGTH)
X_test, y_test = generate_sequences(test_sequences,SEQUENCE_LENGTH)

In [88]:
print(f"Training Samples   : {len(X_train):,}")
print(f"Validation Samples : {len(X_valid):,}")
print(f"Test Samples       : {len(X_test):,}")

Training Samples   : 1,898,620
Validation Samples : 198,603
Test Samples       : 223,641


In [89]:
sample = 0
print("Input IDs:\n")
print(X_train[sample])
print("\nTarget ID:\n")
print(y_train[sample])

Input IDs:

[20628, 127, 3908, 90, 43, 44563, 4419, 23, 752, 43, 27246, 3, 7036, 4, 3908, 5, 2, 23380, 90, 22]

Target ID:

3


In [90]:
index_word = tokenizer.index_word
sample = 9
decoded_input = [index_word.get(idx, "<UNK>") for idx in X_train[sample]]
decoded_target = index_word.get(y_train[sample],"<UNK>")

print("Input:\n")
print(decoded_input)
print("\nTarget:\n")
print(decoded_target)

Input:

[':', '戦場のヴァルキュリア3', ',', 'lit', '.', 'Valkyria', 'of', 'the', 'Battlefield', '3', ')', ',', 'commonly', 'referred', 'to', 'as', 'Valkyria', 'Chronicles', 'III', 'outside']

Target:

Japan


In [91]:
mlflow.log_metrics({
    "train_samples": len(X_train),
    "valid_samples": len(X_valid),
    "test_samples": len(X_test),
    "sequence_length": SEQUENCE_LENGTH
})

In [92]:
X_train = np.array(X_train, dtype=np.int32)
y_train = np.array(y_train, dtype=np.int32)

X_valid = np.array(X_valid, dtype=np.int32)
y_valid = np.array(y_valid, dtype=np.int32)

X_test = np.array(X_test, dtype=np.int32)
y_test = np.array(y_test, dtype=np.int32)

In [93]:
assert X_train.max() < vocab_size, f"X_train has index {X_train.max()} >= vocab_size {vocab_size}"
assert y_train.max() < vocab_size, f"y_train has index {y_train.max()} >= vocab_size {vocab_size}"

In [94]:
print("=" * 60)
print("Dataset Shapes")
print("=" * 60)

print(f"X_train : {X_train.shape}")
print(f"y_train : {y_train.shape}")

print(f"\nX_valid : {X_valid.shape}")
print(f"y_valid : {y_valid.shape}")

print(f"\nX_test  : {X_test.shape}")
print(f"y_test  : {y_test.shape}")


Dataset Shapes
X_train : (1898620, 20)
y_train : (1898620,)

X_valid : (198603, 20)
y_valid : (198603,)

X_test  : (223641, 20)
y_test  : (223641,)


In [95]:
print(f"X_train Memory : {X_train.nbytes / 1024**2:.2f} MB")
print(f"y_train Memory : {y_train.nbytes / 1024**2:.2f} MB")

print(f"X_valid Memory : {X_valid.nbytes / 1024**2:.2f} MB")
print(f"y_valid Memory : {y_valid.nbytes / 1024**2:.2f} MB")

print(f"X_test Memory  : {X_test.nbytes / 1024**2:.2f} MB")
print(f"y_test Memory  : {y_test.nbytes / 1024**2:.2f} MB")

X_train Memory : 144.85 MB
y_train Memory : 7.24 MB
X_valid Memory : 15.15 MB
y_valid Memory : 0.76 MB
X_test Memory  : 17.06 MB
y_test Memory  : 0.85 MB


In [96]:
mlflow.log_metrics({
    "num_train_samples": X_train.shape[0],
    "num_valid_samples": X_valid.shape[0],
    "num_test_samples": X_test.shape[0],

    "sequence_length": X_train.shape[1],

    "vocab_size": vocab_size
})

In [97]:
BATCH_SIZE = config["batch_size"]
train_dataset = (
    tf.data.Dataset
    .from_tensor_slices((X_train, y_train))
    .shuffle(
        buffer_size=len(X_train),
        seed=SEED,
        reshuffle_each_iteration=True
    )
    .batch(BATCH_SIZE)
    .cache()
    .prefetch(tf.data.AUTOTUNE)
)

In [98]:
valid_dataset = (
    tf.data.Dataset
    .from_tensor_slices((X_valid, y_valid))
    .batch(BATCH_SIZE)
    .cache()
    .prefetch(tf.data.AUTOTUNE)
)

In [99]:
test_dataset = (
    tf.data.Dataset
    .from_tensor_slices((X_test, y_test))
    .batch(BATCH_SIZE)
    .cache()
    .prefetch(tf.data.AUTOTUNE)
)

In [100]:
for batch_x, batch_y in train_dataset.take(1):
    print("Input Shape :", batch_x.shape)
    print("Target Shape:", batch_y.shape)

    print("\nFirst Input Sequence:")
    print(batch_x[0].numpy())

    print("\nFirst Target:")
    print(batch_y[0].numpy())

Input Shape : (64, 20)
Target Shape: (64,)

First Input Sequence:
[  12  194  170    3 1760    2  602  775    4 2115 1484    6 1103    3
 1749  693   45  417 6233   21]

First Target:
244


In [101]:
mlflow.log_params({"batch_size": BATCH_SIZE})

mlflow.log_metrics({
    "train_batches": len(train_dataset),
    "valid_batches": len(valid_dataset),
    "test_batches": len(test_dataset)
})

# Making a baseline model to test different preprocessing experimnts, will improve the architecture later on

In [102]:
# Defining some contsnats
VOCAB_SIZE = vocab_size
EMBEDDING_DIM = config["embedding_dim"]
LSTM_UNITS = config["lstm_units"]
SEQUENCE_LENGTH = config["sequence_length"]
LEARNING_RATE = config["learning_rate"]

In [103]:
model = Sequential()
model.add(Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=SEQUENCE_LENGTH, name="embedding"))
model.add(LSTM(units=LSTM_UNITS, name="lstm"))
model.add(Dense(units=VOCAB_SIZE, activation="softmax", name="output"))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [104]:
optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])

In [105]:
model.build(input_shape=(None, SEQUENCE_LENGTH))

In [106]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 20, 128)        │     9,807,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 256)            │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 76618)          │    19,690,826 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 29,892,170 (114.03 MB)

 Trainable params: 29,892,170 (114.03 MB)

 Non-trainable params: 0 (0.00 B)

In [107]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

checkpoint = ModelCheckpoint(
    "best_model.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

In [108]:
history = model.fit(train_dataset, validation_data=valid_dataset, epochs=config["epochs"],callbacks=[ early_stopping, reduce_lr, checkpoint], verbose=1)

Epoch 1/10
29666/29666 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.1459 - loss: 6.7704
Epoch 1: val_loss improved from None to 6.10637, saving model to best_model.keras

Epoch 1: finished saving model to best_model.keras
29666/29666 ━━━━━━━━━━━━━━━━━━━━ 1106s 37ms/step - accuracy: 0.1791 - loss: 6.2832 - val_accuracy: 0.2065 - val_loss: 6.1064 - learning_rate: 0.0010
Epoch 2/10
29666/29666 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.2173 - loss: 5.5285
Epoch 2: val_loss improved from 6.10637 to 6.07560, saving model to best_model.keras

Epoch 2: finished saving model to best_model.keras
29666/29666 ━━━━━━━━━━━━━━━━━━━━ 1097s 37ms/step - accuracy: 0.2244 - loss: 5.4510 - val_accuracy: 0.2133 - val_loss: 6.0756 - learning_rate: 0.0010
Epoch 3/10
29665/29666 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.2414 - loss: 5.1182
Epoch 3: val_loss did not improve from 6.07560
29666/29666 ━━━━━━━━━━━━━━━━━━━━ 1086s 37ms/step - accuracy: 0.2472 - loss: 5.0522 - val_accuracy: 0.2114 - val

2026/07/26 15:09:38 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during tensorflow autologging: INVALID_PARAMETER_VALUE: Response: {'error_code': 'INVALID_PARAMETER_VALUE'}


In [109]:
best_model = tf.keras.models.load_model("best_model.keras")
test_loss, test_accuracy = best_model.evaluate(test_dataset)
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.4f}")
perplexity = np.exp(test_loss)
print(f"Test Perplexity: {perplexity:.2f}")

3495/3495 ━━━━━━━━━━━━━━━━━━━━ 65s 18ms/step - accuracy: 0.2145 - loss: 6.1288
Test Loss     : 6.1288
Test Accuracy : 0.2145
Test Perplexity: 458.86
